# AI‑Driven Self‑Serve Analytical Intelligence Platform (MVP)

This notebook demonstrates an end‑to‑end prototype of an AI‑powered self‑serve analytics system.

**Key capabilities**
- Natural Language → Structured Intent
- Guideline‑driven Hypothesis Generation
- Schema Retrieval using Embeddings (RAG)
- Governed NL2SQL Generation
- DuckDB Query Execution
- Driver & Anomaly Analysis
- Transparent Final Answer with Lineage & Confidence

In [ ]:
!pip install duckdb pandas numpy faker sentence-transformers

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import random
from faker import Faker
from datetime import datetime, timedelta
from sentence_transformers import SentenceTransformer


In [ ]:
fake = Faker()
np.random.seed(42)

N_ROWS = 50000
START_DATE = datetime.now() - timedelta(days=365)

payment_methods = ["UPI", "CARD", "WALLET"]
statuses = ["initiated", "failed", "cleared"]
regions = ["NORTH", "SOUTH", "EAST", "WEST"]
failure_reasons = ["timeout", "bank_error", "insufficient_funds"]

data = []

for _ in range(N_ROWS):
    date = START_DATE + timedelta(days=random.randint(0, 364))
    status = random.choices(statuses, weights=[0.1, 0.2, 0.7])[0]

    data.append({
        "transaction_id": fake.uuid4(),
        "transaction_date": date.date(),
        "payment_method": random.choices(payment_methods, weights=[0.6, 0.25, 0.15])[0],
        "transaction_status": status,
        "net_settlement_amount": round(random.uniform(10, 5000), 2) if status == "cleared" else 0,
        "region": random.choice(regions),
        "failure_reason": random.choice(failure_reasons) if status == "failed" else None,
        "customer_id": fake.uuid4(),   # PII
        "country": "IN"
    })

df_transactions = pd.DataFrame(data)
df_transactions.head()


In [ ]:
con = duckdb.connect(database=":memory:")
con.execute("CREATE TABLE transactions AS SELECT * FROM df_transactions")


In [ ]:
BUSINESS_GLOSSARY = {
    "revenue": {
        "expression": "SUM(net_settlement_amount)",
        "mandatory_filters": {"transaction_status": "cleared"},
        "description": "Net settled revenue for cleared transactions"
    },
    "failed_transactions": {
        "expression": "COUNT(transaction_id)",
        "mandatory_filters": {"transaction_status": "failed"},
        "description": "Number of failed transactions"
    }
}

GOVERNANCE = {
    "pii_columns": ["customer_id"],
    "row_level_filters": {"country": "IN"}
}

In [ ]:
SCHEMA_CATALOG = {
    "transactions": {
        "columns": {
            "transaction_date": "Date of transaction",
            "payment_method": "Payment mode such as UPI or CARD",
            "transaction_status": "Transaction lifecycle status (initiated, failed, cleared)",
            "net_settlement_amount": "Final settled transaction amount after fees",
            "region": "Geographical business region (NORTH, SOUTH, EAST, WEST)",
            "failure_reason": "Reason for transaction failure, if applicable"
        }
    }
}


In [ ]:
def understand_query(question: str):
    return {
        "original_question": question,
        "intent_type": "temporal_comparison" if "change" in question.lower() else "aggregate_metric",
        "metric": "revenue",
        "dimensions": ["month"],
        "time_range": "last_month",
        "assumptions": ["Monthly aggregation assumed"],
        "ambiguities": []
    }

In [ ]:
ALLOWED_HYPOTHESIS_TYPES = [
    "aggregate_metric",
    "temporal_comparison",
    "dimensional_segmentation",
    "driver_analysis",
    "cohort_analysis",
    "anomaly_explanation"
]

def generate_hypotheses(intent):
    hypotheses = []

    if intent["intent_type"] == "aggregate_metric":
        hypotheses.append({
            "id": "H1",
            "type": "aggregate_metric",
            "description": "Compute total revenue",
            "metric": intent["metric"],
            "status": "candidate"
        })

    if intent["intent_type"] == "temporal_comparison":
        hypotheses.append({
            "id": "H2",
            "type": "temporal_comparison",
            "description": "Compare revenue month over month",
            "metric": intent["metric"],
            "comparison": "MoM",
            "status": "selected"
        })

    return hypotheses

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

schema_docs = []
schema_index = []

for table, meta in SCHEMA_CATALOG.items():
    for col, desc in meta["columns"].items():
        doc = f"Table {table}, Column {col}: {desc}"
        schema_docs.append(doc)
        schema_index.append({"table": table, "column": col})

schema_embeddings = embedding_model.encode(schema_docs, normalize_embeddings=True)

def retrieve_relevant_schema(intent, top_k=5):
    query = f"{intent['metric']} {intent['intent_type']} {intent['time_range']}"
    q_emb = embedding_model.encode([query], normalize_embeddings=True)
    scores = np.dot(schema_embeddings, q_emb.T).squeeze()
    top_idx = scores.argsort()[-top_k:][::-1]
    return [schema_index[i] for i in top_idx]

In [ ]:
def generate_sql(hypothesis):
    metric_def = BUSINESS_GLOSSARY[hypothesis["metric"]]

    sql = f"""
    SELECT
        strftime(transaction_date, '%Y-%m') AS month,
        {metric_def['expression']} AS revenue
    FROM transactions
    WHERE transaction_status = 'cleared'
      AND country = 'IN'
    GROUP BY month
    ORDER BY month
    """

    lineage = {
        "tables_used": ["transactions"],
        "columns_used": ["transaction_date", "net_settlement_amount"],
        "filters": ["transaction_status=cleared", "country=IN"]
    }

    return sql.strip(), lineage

In [ ]:
def execute_query(sql):
    return con.execute(sql).df()

In [ ]:
def generate_driver_analysis_sql(metric, dimension):
    metric_def = BUSINESS_GLOSSARY[metric]
    return f"""
    SELECT
        {dimension},
        {metric_def['expression']} AS contribution
    FROM transactions
    WHERE transaction_status = 'cleared'
      AND country = 'IN'
    GROUP BY {dimension}
    ORDER BY contribution DESC
    """.strip()


In [ ]:
def detect_anomalies(df, threshold=2.0):
    mean = df["revenue"].mean()
    std = df["revenue"].std()
    df["z_score"] = (df["revenue"] - mean) / std
    return df[df["z_score"].abs() > threshold]

In [ ]:
def synthesize_answer(df):
    pct = df["revenue"].pct_change().iloc[-1] * 100
    answer = f"Revenue changed by {pct:.2f}% month‑over‑month."
    caveats = [
        "Only cleared transactions included",
        "Monthly aggregation assumed"
    ]
    confidence = 0.85 if len(df) > 2 else 0.6
    return answer, caveats, confidence


In [ ]:
question = "How did revenue change last month?"

intent = understand_query(question)
hypotheses = generate_hypotheses(intent)
selected_hypothesis = next(h for h in hypotheses if h["status"] == "selected")

schema_used = retrieve_relevant_schema(intent)
sql, lineage = generate_sql(selected_hypothesis)
result = execute_query(sql)

driver_sql = generate_driver_analysis_sql("revenue", "region")
driver_result = execute_query(driver_sql)

anomalies = detect_anomalies(result)
answer, caveats, confidence = synthesize_answer(result)

FINAL_RESPONSE = {
    "question": question,
    "intent": intent,
    "hypotheses": hypotheses,
    "selected_hypothesis": selected_hypothesis,
    "schema_retrieved": schema_used,
    "generated_sql": sql,
    "query_result_preview": result.head(),
    "driver_analysis": driver_result.head(5).to_dict(orient="records"),
    "anomalies_detected": anomalies.to_dict(orient="records"),
    "answer": answer,
    "confidence": confidence,
    "caveats": caveats,
    "lineage": lineage
}

FINAL_RESPONSE
